# Multi-Target DTI Prediction: Moringa oleifera Phytochemicals vs MAO-B, AChE, BACE1

Follow-up study to single-target MAO-B docking + ADMET work. This notebook:
1. Fetches target protein sequences from UniProt
2. Fetches compound SMILES from PubChem (by name) — edit the list below with your exact 13 compounds
3. Runs DeepPurpose pretrained DTI models across all compound x target pairs
4. Exports a results table (CSV) you can bring into your paper

**Run cells top to bottom, one at a time (Shift+Enter).** Set Runtime > Change runtime type > GPU first.

## 1. Install packages (run once per session)

In [ ]:
!git clone https://github.com/kexinhuang12345/DeepPurpose.git
%cd /content/DeepPurpose
!pip install -e . -q
!pip install rdkit pubchempy -q
!pip install git+https://github.com/bp-kelley/descriptastorus -q

## 2. Fetch target protein sequences from UniProt

In [ ]:
import requests

targets = {
    "MAOB": "P27338",
    "AChE": "P22303",
    "BACE1": "P56817"
}

sequences = {}
for name, acc in targets.items():
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    r = requests.get(url)
    fasta = r.text
    seq = "".join(fasta.split("\n")[1:]).strip()
    sequences[name] = seq
    print(f">{name}|{acc}  length={len(seq)}")
    print(seq[:60] + "...\n")

## 3. Fetch compound SMILES from PubChem

**Edit this list** so it exactly matches the 13 compounds (and Selegiline) from your MAO-B paper. Using compound *names* here and pulling SMILES live from PubChem avoids manual transcription errors — but double check each fetched SMILES against the structure/CID you used in your SwissADME work, to keep both papers consistent.

In [ ]:
import pubchempy as pcp

compound_names = [
    "3-p-Coumaroylquinic acid",
    "Isolariciresinol",
    "4-Caffeoylquinic acid",
    "Rutin",
    "Luteolin",
    "Kaempferol",
    "Quercetin",
    "Secoisolariciresinol",
    "Apigenin",
    "Myricetin",
    "Medioresinol",
    "O-Coumaric Acid",
    "Selegiline"
]

drugs = {}
failed = []
for name in compound_names:
    try:
        result = pcp.get_compounds(name, 'name')
        if result:
            drugs[name] = result[0].canonical_smiles
            print(f"{name}: {result[0].canonical_smiles}  (CID {result[0].cid})")
        else:
            failed.append(name)
    except Exception as e:
        failed.append(name)
        print(f"FAILED: {name} -> {e}")

if failed:
    print("\nCould not resolve automatically, add SMILES manually for:", failed)

If any names fail to resolve (common for less-standard names like some conjugated acids), look up the compound manually on pubchem.ncbi.gov, copy its Canonical SMILES, and add it directly:
```python
drugs["3-p-Coumaroylquinic acid"] = "PASTE_SMILES_HERE"
```

## 4. Build the full compound x target pair list

In [ ]:
drug_list, target_list, drug_names, target_names = [], [], [], []

for dname, smiles in drugs.items():
    for tname, tseq in sequences.items():
        drug_list.append(smiles)
        target_list.append(tseq)
        drug_names.append(dname)
        target_names.append(tname)

print(f"Total pairs to predict: {len(drug_list)}  ({len(drugs)} compounds x {len(sequences)} targets)")

## 5. Run DeepPurpose pretrained DTI model

In [ ]:
from DeepPurpose import utils, DTI

X_pred = utils.data_process(
    X_drug=drug_list,
    X_target=target_list,
    y=[0]*len(drug_list),   # placeholder required by the API, not used for prediction
    drug_encoding='Morgan',
    target_encoding='CNN',
    split_method='no_split'
)

model = DTI.model_pretrained(model='Morgan_CNN_BindingDB')
y_pred = model.predict(X_pred)

print("Done. Number of predictions:", len(y_pred))

## 6. Organize results into a table and export CSV

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Compound": drug_names,
    "Target": target_names,
    "Predicted_Score": y_pred
})

# Pivot so each row = compound, columns = targets, easy to read/compare
pivot = results_df.pivot(index="Compound", columns="Target", values="Predicted_Score")
print(pivot)

results_df.to_csv("DTI_predictions_long.csv", index=False)
pivot.to_csv("DTI_predictions_by_target.csv")
print("\nSaved: DTI_predictions_long.csv and DTI_predictions_by_target.csv")

## 7. Download the results to your computer

In [ ]:
from google.colab import files
files.download("DTI_predictions_long.csv")
files.download("DTI_predictions_by_target.csv")

## Notes for your methods section

- Target sequences: canonical UniProt entries, MAOB (P27338), AChE (P22303), BACE1 (P56817), retrieved via UniProt REST API.
- Compound structures: canonical SMILES retrieved from PubChem, cross-checked against structures used in the prior MAO-B docking/ADMET study for consistency.
- Drug encoding: Morgan (ECFP) fingerprint. Target encoding: CNN. Pretrained model: Morgan_CNN_BindingDB (DeepPurpose, Huang et al.).
- Check the DeepPurpose repo/model card for the exact scale of the output (binding affinity proxy vs interaction probability) before writing units into your Results section — this can vary by pretrained checkpoint.